# 2024 API prototype

This notebook is a self-contained API experiment that traces a single organization across the
three public money/policy sources for the **2024 cycle** and stitches them together:

```
LDA lobbying filings  --(name)-->  FEC committee / PAC  --(money)-->  disbursements
        |                                                                   
        +--(bills named in lobbying text)-->  Congress.gov bills
```

Use `01_build_dataset.ipynb` and `02_review_and_explore.ipynb` for project analysis.
They persist evidence and require review before cross-source organization names become joins.

| Source | 2024 selector | Join key out |
|--------|---------------|--------------|
| Senate LDA | `filing_year = 2024` | `client_name` → `org_key` |
| FEC | `cycle = 2024`, `two_year_transaction_period = 2024` | `committee.name` → `org_key` |
| Congress.gov | 118th Congress (2023–24) | `(bill_type, number)` from LDA text |

> Run top-to-bottom. Only `DEMO_KEY` is needed to start; add real keys in the **Config** cell
> to raise rate limits and enable Congress.gov bill titles.


In [1]:
# === Imports & config (run once) ===
import re
import requests
import pandas as pd
from collections import Counter
from difflib import SequenceMatcher

# --- 2024 cycle selectors ---
FILING_YEAR = 2024          # LDA filing year
CYCLE = 2024                # FEC two-year transaction period (covers 2023-2024)
CONGRESS = 118              # 118th Congress = 2023-2024

# --- API keys (DEMO_KEY works but is heavily rate-limited) ---
FEC_API_KEY = "DEMO_KEY"    # real key: https://api.open.fec.gov/developers/
CONGRESS_API_KEY = ""       # free key: https://api.congress.gov/sign-up/  (enables bill titles)

# --- Matching ---
MATCH_THRESHOLD = 0.60      # min normalized-name similarity to accept an LDA<->FEC match


## 1. Entity resolution — collapse org names to a join key

The sources name the same organization differently (`PFIZER INC`, `PFIZER, INC.`,
`PFIZER INC. PAC`). Normalize each to a canonical `org_key` so they can be merged.


In [2]:
# === Entity resolution helpers ===
# Legal / structural tokens that add noise but no identity.
_NOISE = {
    "inc", "incorporated", "llc", "lp", "llp", "co", "corp", "corporation",
    "company", "the", "and", "of", "for", "fund", "foundation", "trust",
    "pac", "committee", "cmte", "political", "action", "associates", "assn",
    "association", "society", "institute", "group", "holdings",
}

def normalize_org_name(name: str) -> str:
    """Canonicalize an org name so equivalent names collide.
    'Pfizer, Inc.' / 'PFIZER INC. PAC' / 'Pfizer Co' -> 'pfizer'."""
    if not name:
        return ""
    name = re.sub(r"[^a-z0-9\s]", " ", name.lower())   # lower + drop punctuation
    return " ".join(t for t in name.split() if t not in _NOISE).strip()

def name_sim(a: str, b: str) -> float:
    """0..1 similarity on normalized names (stdlib; swap for rapidfuzz in prod)."""
    return SequenceMatcher(None, normalize_org_name(a), normalize_org_name(b)).ratio()


## 2. Source fetchers (2024)

Thin wrappers over the three live APIs, all scoped to the 2024 cycle. Each returns
plain JSON/lists so the pipeline below stays easy to read.


In [3]:
# === Source fetchers ===
def fetch_lda_filings(org, year=FILING_YEAR, page_size=25):
    """Senate LDA filings for an org (client) in a given year, with lobbying activities."""
    r = requests.get(
        "https://lda.senate.gov/api/v1/filings/",
        params={"client_name": org, "filing_year": year, "page_size": page_size},
        timeout=30,
    )
    r.raise_for_status()
    return r.json().get("results", [])

def fetch_fec_committees(org, cycle=CYCLE, per_page=20):
    """FEC committees whose name matches `org`, active in the given 2-year cycle."""
    r = requests.get(
        "https://api.open.fec.gov/v1/committees",
        params={"api_key": FEC_API_KEY, "q": org, "cycle": cycle, "per_page": per_page},
        timeout=30,
    )
    r.raise_for_status()
    return r.json().get("results", [])

def fetch_fec_disbursements(committee_id, cycle=CYCLE, per_page=20):
    """Schedule B disbursements (money the PAC spent) for a committee in the cycle."""
    r = requests.get(
        "https://api.open.fec.gov/v1/schedules/schedule_b",
        params={"api_key": FEC_API_KEY, "committee_id": committee_id,
                "two_year_transaction_period": cycle, "per_page": per_page,
                "sort": "-disbursement_amount"},
        timeout=30,
    )
    r.raise_for_status()
    return r.json().get("results", [])

# Bill-reference regex: longer types first so "hjres" wins over "h". (cf. analysis/bills.py)
_BILL_RE = re.compile(
    r"\b(H\.?\s?J\.?\s?Res\.?|S\.?\s?J\.?\s?Res\.?|"
    r"H\.?\s?Con\.?\s?Res\.?|S\.?\s?Con\.?\s?Res\.?|"
    r"H\.?\s?Res\.?|S\.?\s?Res\.?|H\.?\s?R\.?|S)\.?\s?(\d{1,5})\b",
    re.IGNORECASE,
)

def extract_bill_keys(text):
    """Yield canonical bill keys like 'hr3684', 's1339' from free text."""
    for m in _BILL_RE.finditer(text or ""):
        yield re.sub(r"[.\s]", "", m.group(1)).lower() + m.group(2)

def fetch_bill(bill_key, congress=CONGRESS):
    """Optional: bill title/action from Congress.gov (needs CONGRESS_API_KEY)."""
    if not CONGRESS_API_KEY:
        return None
    btype, bnum = re.match(r"([a-z]+)(\d+)", bill_key).groups()
    r = requests.get(
        f"https://api.congress.gov/v3/bill/{congress}/{btype}/{bnum}",
        params={"api_key": CONGRESS_API_KEY, "format": "json"}, timeout=30,
    )
    return r.json().get("bill") if r.ok else None


## 3. The pipeline — `trace_org()`

Ties one organization across all sources for 2024:

1. **Lobbying** — pull LDA filings; coalesce `income`/`expenses` into one `lobbying_usd`, aggregated per `org_key` (so we don't fan-out rows).
2. **Money** — fuzzy-match the org's name to FEC committees (`name_sim >= MATCH_THRESHOLD`, so `EXXONMOBIL` still matches `Exxon Mobil`), then pull each PAC's disbursements.
3. **Policy** — extract bill numbers named in the lobbying text (the link to Congress).

Returns a `summary` dict plus the supporting DataFrames.


In [8]:
# === trace_org: stitch one organization across LDA + FEC + Congress (2024) ===
def trace_org(org):
    filings = fetch_lda_filings(org)
    committees = fetch_fec_committees(org)

    # 1) LOBBYING: coalesce income/expenses, aggregate per org_key (one row per org)
    lda_rows = []
    for f in filings:
        name = (f.get("client") or {}).get("name")
        if not name:
            continue
        usd = f.get("income") or f.get("expenses") or 0      # filers report one or the other
        lda_rows.append({"org_key": normalize_org_name(name),
                         "client": name, "lobbying_usd": float(usd or 0)})
    lda_df = pd.DataFrame(lda_rows)
    lda_agg = (lda_df.groupby("org_key", as_index=False)
                     .agg(client=("client", "first"),
                          n_filings=("client", "size"),
                          lobbying_usd=("lobbying_usd", "sum"))
               if not lda_df.empty else lda_df)

    # 2) MONEY: FEC committees -> match on org_key -> pull each PAC's disbursements
    # Fuzzy-match each committee against the org's lobbying-client name(s); keep the
    # ones above MATCH_THRESHOLD. This tolerates spacing/punctuation variants that an
    # exact key join would miss (EXXONMOBIL vs EXXON MOBIL).
    client_names = list(lda_agg["client"]) if not lda_agg.empty else [org]
    fec_df = pd.DataFrame([{"committee_id": c["committee_id"],
                            "committee": c["name"],
                            "type": c.get("committee_type")} for c in committees])
    if not fec_df.empty:
        fec_df["match_score"] = fec_df["committee"].apply(
            lambda n: max(name_sim(n, cn) for cn in client_names))
        matched = (fec_df[fec_df["match_score"] >= MATCH_THRESHOLD]
                   .drop_duplicates("committee_id")
                   .sort_values("match_score", ascending=False))
    else:
        matched = fec_df

    disb_rows = []
    for cid in matched["committee_id"]:
        for d in fetch_fec_disbursements(cid):
            disb_rows.append({"committee_id": cid,
                              "recipient": d.get("recipient_name"),
                              "amount": float(d.get("disbursement_amount") or 0),
                              "date": d.get("disbursement_date"),
                              "purpose": d.get("disbursement_description")})
    disb_df = pd.DataFrame(disb_rows)

    # 3) POLICY: bills named in the lobbying-activity text -> link to Congress
    bill_counts = Counter()
    for f in filings:
        for act in (f.get("lobbying_activities") or []):
            for key in set(extract_bill_keys(act.get("description"))):
                bill_counts[key] += 1
    bills_df = pd.DataFrame(bill_counts.most_common(),
                            columns=["bill_key", "filings_mentioning"])

    summary = {
        "org": org,
        "n_filings": len(filings),
        "total_lobbying_usd": float(lda_agg["lobbying_usd"].sum()) if not lda_agg.empty else 0.0,
        "n_pacs": int(matched["committee_id"].nunique()),
        "pac_total_disbursed": float(disb_df["amount"].sum()) if not disb_df.empty else 0.0,
        "n_bills_lobbied": len(bills_df),
    }
    return {"summary": summary, "lobbying": lda_agg, "pacs": matched,
            "disbursements": disb_df, "bills": bills_df}

## 4. Run it — trace one organization

Change `ORG` to any company / trade group. The output is the full 2024 picture:
lobbying spend, the matched PAC(s), where that PAC's money went, and which bills were lobbied.


In [5]:
# === Trace one org across all sources (2024) ===
ORG = "Pfizer"
result = trace_org(ORG)

print("SUMMARY:", result["summary"], "\n")

print("Lobbying (LDA, aggregated per org):")
display(result["lobbying"])

print("Matched PAC(s) (FEC):")
display(result["pacs"])

print("Where the PAC money went (top disbursements):")
display(result["disbursements"].head(10))

print("Bills lobbied (named in the lobbying text -> Congress):")
display(result["bills"].head(15))

# Optional: enrich the top bill with its title from Congress.gov (needs CONGRESS_API_KEY).
if CONGRESS_API_KEY and len(result["bills"]):
    top = result["bills"].iloc[0]["bill_key"]
    bill = fetch_bill(top)
    if bill:
        print(f"\nTop bill {top.upper()}: {bill.get('title')}")
        print("latest action:", (bill.get("latestAction") or {}).get("text"))


SUMMARY: {'org': 'Pfizer', 'n_filings': 25, 'total_lobbying_usd': 6880000.0, 'n_pacs': 1, 'pac_total_disbursed': 275305.0, 'n_bills_lobbied': 31} 

Lobbying (LDA, aggregated per org):


,org_key,client,n_filings,lobbying_usd
0,pfizer,PFIZER INC,25,6880000.0


Matched PAC(s) (FEC):


,org_key,committee_id,committee,type
0,pfizer,C00016683,PFIZER INC. PAC,Q


Where the PAC money went (top disbursements):


,committee_id,recipient,amount,date,purpose
0,C00016683,PFIZER MISSOURI POLITICAL ACTION COMMITTEE - F...,21500.0,2023-10-19,NONFEDERAL CONTRIBUTION
1,C00016683,MCCARTHY VICTORY FUND,20000.0,2023-03-21,2023 CONTRIBUTION
2,C00016683,PFIZER MISSOURI POLITICAL ACTION COMMITTEE - F...,16625.0,2024-09-19,NONFEDERAL CONTRIBUTION
3,C00016683,REPUBLICAN STATE LEADERSHIP COMMITTEE (RSLC),16000.0,2023-11-17,NONFEDERAL CONTRIBUTION
4,C00016683,REPUBLICAN NATIONAL COMMITTEE,15000.0,2023-04-18,2023 CONTRIBUTION
5,C00016683,DNC SERVICES CORPORATION/DEMOCRATIC NATIONAL C...,15000.0,2023-04-18,2023 CONTRIBUTION
6,C00016683,NRSC,15000.0,2023-03-21,2023 CONTRIBUTION
7,C00016683,NRCC,15000.0,2023-03-21,None
8,C00016683,DSCC,15000.0,2023-03-21,2023 CONTRIBUTION
9,C00016683,DCCC,15000.0,2023-03-21,2023 CONTRIBUTION


Bills lobbied (named in the lobbying text -> Congress):


,bill_key,filings_mentioning
0,hr5378,4
1,hr1458,3
2,s127,2
3,s2333,2
4,hr2679,2
5,hr4822,2
6,hr3290,2
7,s1339,2
8,s113,2
9,s148,2


## 5. Batch — compare several organizations (2024)

Run the pipeline over a watchlist and rank by lobbying spend. This is the headline
2024 table: who lobbied the most, how many PACs they run, how much those PACs spent,
and how many bills they touched.

> `DEMO_KEY` is rate-limited (~hundreds/hr). Keep the list short or add a real `FEC_API_KEY`.


In [9]:
# === Batch: trace a watchlist and rank by 2024 lobbying spend ===
WATCHLIST = ["Pfizer", "Lockheed Martin", "Exxon Mobil"]

summary_rows = []
for org in WATCHLIST:
    try:
        summary_rows.append(trace_org(org)["summary"])
    except Exception as exc:                      # keep going if one org rate-limits / errors
        print(f"skip {org}: {exc}")

summary_df = (pd.DataFrame(summary_rows)
                .sort_values("total_lobbying_usd", ascending=False)
                .reset_index(drop=True))
display(summary_df)


skip Lockheed Martin: 429 Client Error: Too Many Requests for url: https://api.open.fec.gov/v1/committees?api_key=DEMO_KEY&q=Lockheed+Martin&cycle=2024&per_page=20
skip Exxon Mobil: 429 Client Error: Too Many Requests for url: https://api.open.fec.gov/v1/committees?api_key=DEMO_KEY&q=Exxon+Mobil&cycle=2024&per_page=20


,org,n_filings,total_lobbying_usd,n_pacs,pac_total_disbursed,n_bills_lobbied
0,Pfizer,25,6880000.0,1,275305.0,31
